# Rollout Sweep Plots: t_density vs frac

In [1]:
import json
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

REPO_BASE = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
while REPO_BASE.name != "persona-shattering-lasr" and REPO_BASE != REPO_BASE.parent:
    REPO_BASE = REPO_BASE.parent

HF_REPO = "persona-shattering-lasr/monorepo"

DATASETS = {
    "t_avoiding_activation_capping": {
        "hf_subdir": (
            "fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding"
            "/t_avoiding-train-20260310-164958/evals/rollout_sweep_activation_capping"
        ),
    },
    "t_avoiding_lora_scaling": {
        "hf_subdir": (
            "fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding"
            "/t_avoiding-train-20260310-164958/evals/rollout_sweep_lora_scaling"
        ),
    },
    "t_enjoying_activation_capping": {
        "hf_subdir": (
            "fine_tuning/llama-3.1-8B-Instruct/toy/t_character_enjoying"
            "/t_enjoying-train-20260312-223656/evals/rollout_sweep_activation_capping"
        ),
    },
    "t_enjoying_lora_scaling": {
        "hf_subdir": (
            "fine_tuning/llama-3.1-8B-Instruct/toy/t_character_enjoying"
            "/t_enjoying-train-20260312-223656/evals/rollout_sweep_lora_scaling"
        ),
    },
}

# Resolve local paths and download missing data
for name, cfg in DATASETS.items():
    hf_subdir = cfg["hf_subdir"]
    data_dir = REPO_BASE / "scratch/monorepo" / hf_subdir
    cfg["data_dir"] = data_dir

    if data_dir.exists() and any(data_dir.iterdir()):
        print(f"[{name}] Data found locally at {data_dir}")
    else:
        print(f"[{name}] Pulling from HuggingFace: {HF_REPO}/{hf_subdir}")
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            allow_patterns=f"{hf_subdir}/**/rollouts_evaluated.jsonl",
            local_dir=data_dir.parents[len(Path(hf_subdir).parts) - 1],
        )
        print(f"[{name}] Download complete.")

[t_avoiding_activation_capping] Data found locally at /Users/anton/dev/LASR/persona-shattering-lasr/scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding/t_avoiding-train-20260310-164958/evals/rollout_sweep_activation_capping
[t_avoiding_lora_scaling] Data found locally at /Users/anton/dev/LASR/persona-shattering-lasr/scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding/t_avoiding-train-20260310-164958/evals/rollout_sweep_lora_scaling
[t_enjoying_activation_capping] Data found locally at /Users/anton/dev/LASR/persona-shattering-lasr/scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_enjoying/t_enjoying-train-20260312-223656/evals/rollout_sweep_activation_capping
[t_enjoying_lora_scaling] Data found locally at /Users/anton/dev/LASR/persona-shattering-lasr/scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_enjoying/t_enjoying-train-20260312-223656/evals/rollout_sweep_lora_scaling


In [2]:
def load_sweep_data(data_dir: Path) -> pd.DataFrame:
    """Load t_density from the final assistant message of each rollout.

    Auto-detects frac_* or scale_* subdirectory naming. Negative values
    are supported (e.g. frac_-0.50, scale_-1.00).

    Returns a DataFrame with columns: sweep_var, category, seed_id, rollout_idx, t_density
    """
    rows = []
    sweep_dirs = sorted(list(data_dir.glob("frac_*")) + list(data_dir.glob("scale_*")))
    for sweep_dir in sweep_dirs:
        sweep_val = float(sweep_dir.name.split("_", 1)[1])
        for cat_dir in sorted(sweep_dir.iterdir()):
            if not cat_dir.is_dir():
                continue
            category = cat_dir.name
            jsonl_path = cat_dir / "rollouts_evaluated.jsonl"
            if not jsonl_path.exists():
                continue
            with open(jsonl_path) as f:
                for line in f:
                    sample = json.loads(line)
                    seed_id = sample["seed_id"]
                    for rollout_idx, msgs in sample["messages"].items():
                        assistant_msgs = [m for m in msgs if m["role"] == "assistant"]
                        if not assistant_msgs:
                            continue
                        final_msg = assistant_msgs[-1]
                        t_density = final_msg.get("scores", {}).get("count_t.density")
                        if t_density is not None:
                            rows.append({
                                "sweep_var": sweep_val,
                                "category": category,
                                "seed_id": seed_id,
                                "rollout_idx": int(rollout_idx),
                                "t_density": t_density,
                            })
    df = pd.DataFrame(rows)
    if df.empty:
        print(f"No data found in {data_dir}")
    else:
        print(f"Loaded {len(df)} datapoints across {df['sweep_var'].nunique()} sweep values and {df['category'].nunique()} categories")
    return df


dfs = {}
for name, cfg in DATASETS.items():
    print(f"\n--- {name} ---")
    dfs[name] = load_sweep_data(cfg["data_dir"])


--- t_avoiding_activation_capping ---
Loaded 16128 datapoints across 21 sweep values and 8 categories

--- t_avoiding_lora_scaling ---
Loaded 17378 datapoints across 23 sweep values and 8 categories

--- t_enjoying_activation_capping ---
Loaded 16128 datapoints across 21 sweep values and 8 categories

--- t_enjoying_lora_scaling ---
Loaded 17664 datapoints across 23 sweep values and 8 categories


In [ ]:
base_colors = {
    "single": (31, 119, 180),     # blue
    "assistant": (255, 127, 14),  # orange
    "aa": (44, 160, 44),          # green
    "user": (214, 39, 40),        # red
}

def rgb(r, g, b, a=1.0):
    return f"rgba({r},{g},{b},{a})"

def lighten(color_tuple, amount=0.4):
    return tuple(int(c + (255 - c) * amount) for c in color_tuple)

suffix_config = {
    "baseline":   {"marker": "circle",         "dash": "solid", "lighten": 0.0},
    "t_enjoying": {"marker": "cross",          "dash": "dot",   "lighten": 0.35},
    "t_avoiding": {"marker": "line-ns-open",   "dash": "dash",  "lighten": 0.35},
}

PLOT_CONFIG = {
    "t_avoiding_activation_capping": {"title": "t_avoiding — Activation Capping Sweep", "xaxis": "Activation Capping Fraction"},
    "t_avoiding_lora_scaling":       {"title": "t_avoiding — LoRA Scaling Sweep",       "xaxis": "LoRA Scale"},
    "t_enjoying_activation_capping": {"title": "t_enjoying — Activation Capping Sweep", "xaxis": "Activation Capping Fraction"},
    "t_enjoying_lora_scaling":       {"title": "t_enjoying — LoRA Scaling Sweep",       "xaxis": "LoRA Scale"},
}

# Compute global y-axis max (mean + std) across all datasets
global_y_max = 0
for df in dfs.values():
    if df.empty:
        continue
    stats_tmp = df.groupby(["sweep_var", "category"])["t_density"].agg(["mean", "std"]).reset_index()
    y_upper = (stats_tmp["mean"] + stats_tmp["std"]).max()
    global_y_max = max(global_y_max, y_upper)

for name, df in dfs.items():
    if df.empty:
        print(f"Skipping {name} (no data)")
        continue

    pcfg = PLOT_CONFIG[name]
    stats = df.groupby(["sweep_var", "category"])["t_density"].agg(["mean", "std", "count"]).reset_index()
    stats["prefix"] = stats["category"].apply(lambda c: c.split("_")[0])
    stats["suffix"] = stats["category"].apply(lambda c: "_".join(c.split("_")[1:]))

    # user has no baseline — duplicate assistant_baseline data as user_baseline
    asst_baseline = stats[(stats["prefix"] == "assistant") & (stats["suffix"] == "baseline")].copy()
    if not asst_baseline.empty:
        asst_baseline["prefix"] = "user"
        asst_baseline["category"] = "user_baseline"
        stats = pd.concat([stats, asst_baseline], ignore_index=True)

    fig = go.Figure()

    for prefix in ["single", "assistant", "aa", "user"]:
        base = base_colors[prefix]
        for suffix, scfg in suffix_config.items():
            mask = (stats["prefix"] == prefix) & (stats["suffix"] == suffix)
            sdf = stats[mask].sort_values("sweep_var")
            if sdf.empty:
                continue

            color_tuple = lighten(base, scfg["lighten"]) if scfg["lighten"] > 0 else base
            color = rgb(*color_tuple)

            fig.add_trace(go.Scatter(
                x=sdf["sweep_var"],
                y=sdf["mean"],
                error_y=dict(type="data", array=sdf["std"].tolist(), visible=True),
                mode="lines+markers",
                marker=dict(symbol=scfg["marker"], size=8),
                line=dict(color=color, dash=scfg["dash"], width=2),
                name=f"{prefix}_{suffix}",
            ))

    fig.update_layout(
        title=f"t Density — {pcfg['title']} (final assistant message)",
        xaxis_title=pcfg["xaxis"],
        yaxis_title="Mean t Density (%)",
        yaxis_range=[0, global_y_max * 1.05],
        legend=dict(x=1.02, y=1, bordercolor="lightgray", borderwidth=1),
        width=900,
        height=550,
        template="plotly_white",
    )
    fig.show()